# Instalacion de dependencias.

In [ ]:
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir unsloth_zoo
!pip install --no-cache-dir --upgrade typing-extensions==4.12.2 pydantic==2.10.6 pydantic-core==2.27.2
!pip install --no-cache-dir trl transformers accelerate datasets bitsandbytes ai-edge-torch litert-lm

# Cargar modelo.

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512

# Cambiamos el modelo a la versión 1B IT
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-3-1b-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    dtype = None,
)

# Mantener r=64 para buena capacidad de adaptación
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

==((====))==  Unsloth 2026.9.2: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
total weights      : 1.862 GiB
no_split classes   : ['Gemma3DecoderLayer', 'SiglipEncoderLayer', 'SiglipMultiheadAttentionPoolingHead', 'SiglipVisionEmbeddings']
output head        : lm_head -> cuda:1
head headroom      : 0.438 GiB
activation reserve : 12.222 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  13.12 GiB  weights  0.900 GiB  free 12.218 GiB  reserve 12.204 GiB
  cuda:1  budget  13.63 GiB  weights  0.962 GiB  free 12.663 GiB  reserve 12.

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


 # Entrenar y probar.

In [ ]:
import torch
import json
import os
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# Configuración principal
max_seq_length = 512
ruta_kaggle = "/kaggle/input/datasets/mikelgorrin/sentences/SpanishSentences.json"
ruta_local = "esdataset.json"

# ==========================================
# PASO 1: CARGAR Y COMBINAR DATOS
# ==========================================
print("📂 Iniciando carga de datos...")
lista_datos = []

# 1. Intentar cargar el archivo local (contiene el progreso de sesiones previas)
if os.path.exists(ruta_local):
    with open(ruta_local, "r", encoding="utf-8") as f:
        contenido = json.load(f)
        if isinstance(contenido, dict):
            lista_datos = contenido.get("root", [])
        elif isinstance(contenido, list):
            lista_datos = contenido
    print(f"¡Cargados {len(lista_datos)} ejemplos desde tu archivo local actualizado!")

# 2. Si no hay archivo local, cargamos el original de Kaggle
elif os.path.exists(ruta_kaggle):
    with open(ruta_kaggle, "r", encoding="utf-8") as f:
        contenido = json.load(f)
        if isinstance(contenido, dict):
            lista_datos = contenido.get("root", [])
        elif isinstance(contenido, list):
            lista_datos = contenido
    print(f"¡Cargados {len(lista_datos)} ejemplos desde el dataset original de Kaggle!")

# 3. Fallback por si ambos archivos están vacíos
if not lista_datos:
    lista_datos = [{"palabras": "ejemplo inicial", "oracion": "Este es un ejemplo de inicialización."}]

dataset_hf = Dataset.from_list(lista_datos)
print(f"Total de ejemplos listos para el entrenamiento: {len(dataset_hf)}\n")

# ==========================================
# PASO 2: CARGAR MODELO BASE
# ==========================================
print("🤖 Cargando modelo base Gemma 3 1B...")
modelo, tokenizador = FastLanguageModel.from_pretrained(
    model_name="google/gemma-3-1b-it",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    dtype=None,
)

# ==========================================
# PASO 3: CONFIGURAR ADAPTADORES LORA
# ==========================================
print("⚙️ Configurando el adaptador LoRA...")
modelo = FastLanguageModel.get_peft_model(
    modelo,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# ==========================================
# PASO 4: FORMATEAR PROMPTS Y ENTRENAR
# ==========================================
def aplicar_formato(fila):
    texto_prompt = f"<start_of_turn>user\nCrea una oración con: {fila['palabras']}.<end_of_turn>\n<start_of_turn>model\nOración: {fila['oracion']}<end_of_turn>"
    return {"text": texto_prompt}

dataset_preparado = dataset_hf.map(aplicar_formato)
total_pasos = max(25, len(dataset_preparado) * 2)

entrenador = SFTTrainer(
    model=modelo,
    tokenizer=tokenizador,
    train_dataset=dataset_preparado,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=total_pasos,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        seed=3407,
        output_dir="salida_entrenamiento",
    ),
)

print("\n🚀 Comenzando el entrenamiento...")
entrenador.train()

modelo.save_pretrained("modelo_gemma_oraciones")
tokenizador.save_pretrained("modelo_gemma_oraciones")
print("\n✅ Modelo entrenado y guardado correctamente.\n")

# ==========================================
# PASO 5: MODO INFERENCIA Y CHAT INTERACTIVO
# ==========================================
print("🔄 Activando modo de inferencia rápida...")
FastLanguageModel.for_inference(modelo)

def generar_texto(palabras_usuario):
    prompt_entrada = f"<start_of_turn>user\nCrea una oración con: {palabras_usuario}.<end_of_turn>\n<start_of_turn>model\nOración:"
    entradas = tokenizador([prompt_entrada], return_tensors="pt").to("cuda")

    salidas = modelo.generate(
        **entradas,
        max_new_tokens=25,
        max_length=None,
        do_sample=False,
        repetition_penalty=1.0,
        eos_token_id=tokenizador.eos_token_id
    )

    texto_decodificado = tokenizador.decode(salidas[0][entradas.input_ids.shape[1]:], skip_special_tokens=True)
    oracion_limpia = texto_decodificado.strip().split('.')[0]
    return oracion_limpia + "." if oracion_limpia else "Error al generar."

print("\n" + "="*50)
print("💬 CHAT INTERACTIVO Y AUTOGUARDADO")
print("1. Escribe tus palabras clave.")
print("2. Presiona ENTER para aceptar la oración generada.")
print("3. Escribe tu propia oración si deseas corregirla.")
print("4. Escribe 'salir' para finalizar y cerrar.")
print("="*50 + "\n")

while True:
    try:
        entrada_palabras = input("🔹 Palabras: ").strip()
    except (KeyboardInterrupt, EOFError):
        break

    if entrada_palabras.lower() == "salir":
        print(f"\n👋 ¡Adiós! Progreso guardado exitosamente en '{ruta_local}'.")
        break
    if not entrada_palabras:
        continue

    respuesta_modelo = generar_texto(entrada_palabras)
    print(f"   🤖 Respuesta: {respuesta_modelo}")

    try:
        correccion_usuario = input("   ✍️ Corrección (Enter si es correcta): ").strip()
    except (KeyboardInterrupt, EOFError):
        break

    oracion_definitiva = correccion_usuario if correccion_usuario else respuesta_modelo
    
    # Añadimos la nueva entrada a nuestra lista en memoria
    lista_datos.append({"palabras": entrada_palabras, "oracion": oracion_definitiva})

    # Guardamos todo el JSON con el formato exacto {"root": [...]}
    with open(ruta_local, "w", encoding="utf-8") as f:
        json.dump({"root": lista_datos}, f, ensure_ascii=False, indent=4)

    if correccion_usuario:
        print(f"   [✔️ Corregido y almacenado en {ruta_local}]\n")
    else:
        print(f"   [✅ Aprobado y almacenado en {ruta_local}]\n")

In [ ]:
import json
import os

ruta_kaggle = "/kaggle/input/datasets/mikelgorrin/sentences/SpanishSentences.json"
ruta_local = "esdataset.json"

lista_total = []

# 1. Cargar el dataset original de Kaggle (que es un JSON con un diccionario "root")
if os.path.exists(ruta_kaggle):
    with open(ruta_kaggle, "r", encoding="utf-8") as f:
        data_kaggle = json.load(f)
        if isinstance(data_kaggle, dict):
            lista_total.extend(data_kaggle.get("root", []))
        elif isinstance(data_kaggle, list):
            lista_total.extend(data_kaggle)

# 2. Cargar tu archivo local con las nuevas interacciones y correcciones
if os.path.exists(ruta_local):
    with open(ruta_local, "r", encoding="utf-8") as f:
        data_local = json.load(f)
        registros_locales = data_local.get("root", []) if isinstance(data_local, dict) else data_local
        
        # Evitar duplicados si alguna frase ya estaba en el de Kaggle
        existentes = {(item.get('palabras'), item.get('oracion')) for item in lista_total}
        for item in registros_locales:
            par = (item.get('palabras'), item.get('oracion'))
            if par not in existentes:
                lista_total.append(item)
                existentes.add(par)

# 3. Estructurar el JSON final combinando ambos con la clave "root"
estructura_json = {"root": lista_total}

# Imprimir en formato JSON bonito por pantalla
print(json.dumps(estructura_json, ensure_ascii=False, indent=4))

# Guardarlo permanentemente como un archivo .json limpio y actualizado
with open("esdataset_final.json", "w", encoding="utf-8") as f:
    json.dump(estructura_json, f, ensure_ascii=False, indent=4)

print(f"\n✅ Archivo combinado guardado como 'esdataset_final.json' con un total de {len(lista_total)} ejemplos.")

# Exportar

In [3]:
import os
import json
import shutil
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from IPython.display import FileLink, display, HTML

# Configuración principal
max_seq_length = 512
ruta_kaggle = "/kaggle/input/datasets/mikelgorrin/sentences/SpanishSentences.json"
ruta_local = "esdataset.json"

# ==========================================
# PASO 1: CARGAR Y COMBINAR DATOS
# ==========================================
print("📂 Iniciando carga de datos...")
lista_datos = []

if os.path.exists(ruta_local):
    with open(ruta_local, "r", encoding="utf-8") as f:
        contenido = json.load(f)
        if isinstance(contenido, dict):
            lista_datos = contenido.get("root", [])
        elif isinstance(contenido, list):
            lista_datos = contenido
    print(f"¡Cargados {len(lista_datos)} ejemplos desde tu archivo local actualizado!")
elif os.path.exists(ruta_kaggle):
    with open(ruta_kaggle, "r", encoding="utf-8") as f:
        contenido = json.load(f)
        if isinstance(contenido, dict):
            lista_datos = contenido.get("root", [])
        elif isinstance(contenido, list):
            lista_datos = contenido
    print(f"¡Cargados {len(lista_datos)} ejemplos desde el dataset original de Kaggle!")

if not lista_datos:
    lista_datos = [{"palabras": "ejemplo inicial", "oracion": "Este es un ejemplo de inicialización."}]

dataset_hf = Dataset.from_list(lista_datos)
print(f"Total de ejemplos listos para el entrenamiento: {len(dataset_hf)}\n")

# ==========================================
# PASO 2: CARGAR MODELO BASE
# ==========================================
print("🤖 Cargando modelo base Gemma 3 1B...")
modelo, tokenizador = FastLanguageModel.from_pretrained(
    model_name="google/gemma-3-1b-it",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    dtype=None,
)

# ==========================================
# PASO 3: CONFIGURAR ADAPTADORES LORA
# ==========================================
print("⚙️ Configurando el adaptador LoRA...")
modelo = FastLanguageModel.get_peft_model(
    modelo,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# ==========================================
# PASO 4: FORMATEAR PROMPTS Y ENTRENAR
# ==========================================
def aplicar_formato(fila):
    texto_prompt = f"<start_of_turn>user\nCrea una oración con: {fila['palabras']}.<end_of_turn>\n<start_of_turn>model\nOración: {fila['oracion']}<end_of_turn>"
    return {"text": texto_prompt}

dataset_preparado = dataset_hf.map(aplicar_formato)
total_pasos = max(25, len(dataset_preparado) * 2)

entrenador = SFTTrainer(
    model=modelo,
    tokenizer=tokenizador,
    train_dataset=dataset_preparado,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=total_pasos,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        seed=3407,
        output_dir="salida_entrenamiento",
    ),
)

print("\n🚀 Comenzando el entrenamiento...")
entrenador.train()

modelo.save_pretrained("modelo_gemma_oraciones")
tokenizador.save_pretrained("modelo_gemma_oraciones")
print("\n✅ Modelo entrenado y guardado correctamente en local.\n")

# ==========================================
# PASO 5: EXPORTAR A FORMATOS GGUF Y LITERT
# ==========================================
print("📦 Guardando modelo fusionado a 16-bit...")
modelo.save_pretrained_merged("modelo_gemma_16bit", tokenizador, save_method="merged_16bit")

print("📦 Exportando a GGUF (Quantización q4_k_m)...")
modelo.save_pretrained_gguf("modelo_gemma_gguf", tokenizador, quantization_method="q4_k_m")

print("📦 Convirtiendo a LiteRT / TFLite mediante litert-torch...")
os.system("pip install -q git+https://github.com/google-ai-edge/litert-torch.git")

try:
    import litert_torch
    litert_torch.convert_from_pytorch(
        model_path="modelo_gemma_16bit",
        output_path="modelo_gemma_litert.tflite"
    )
    print("✅ Exportación a LiteRT completada.")
except Exception as e:
    print(f"⚠️ Nota de conversión a LiteRT: {e}")
    # En caso de no tener litert-torch en el entorno, empaqueta la versión 16bit lista para exportador móvil
    shutil.copytree("modelo_gemma_16bit", "modelo_gemma_litert", dirs_exist_ok=True)

# Compresión para descarga en Kaggle/Colab
shutil.make_archive("modelo_gemma_gguf", 'zip', "modelo_gemma_gguf")
if os.path.exists("modelo_gemma_litert.tflite"):
    shutil.make_archive("modelo_gemma_litert", 'zip', filter_dir=False, base_dir="modelo_gemma_litert.tflite")
else:
    shutil.make_archive("modelo_gemma_litert", 'zip', "modelo_gemma_litert")

# ==========================================
# PASO 6: GENERAR BOTONES DE DESCARGA (GUI)
# ==========================================
display(HTML("""
<div style="background-color: #f0f4f8; padding: 20px; border-radius: 10px; border: 1px solid #d0d7de;">
    <h3 style="color: #0969da; margin-top: 0;">📥 DESCARGA DIRECTA DE MODELOS ENTRENADOS</h3>
    <p>Haz clic en los botones para descargar tu modelo en el formato que necesites:</p>
    <a href="./modelo_gemma_gguf.zip" download style="background-color: #2ea44f; color: white; padding: 10px 20px; text-decoration: none; border-radius: 6px; font-weight: bold; margin-right: 10px; display: inline-block;">
        💾 Descargar Modelo GGUF (.zip)
    </a>
    <a href="./modelo_gemma_litert.zip" download style="background-color: #0969da; color: white; padding: 10px 20px; text-decoration: none; border-radius: 6px; font-weight: bold; display: inline-block;">
        📱 Descargar Modelo LiteRT (.zip)
    </a>
</div>
"""))

# Fallback por consola
print("\nEnlaces alternativos de descarga (FileLink):")
display(FileLink(r'modelo_gemma_gguf.zip'))
display(FileLink(r'modelo_gemma_litert.zip'))

# ==========================================
# PASO 7: MODO INFERENCIA Y CHAT INTERACTIVO
# ==========================================
print("\n🔄 Activando modo de inferencia rápida...")
FastLanguageModel.for_inference(modelo)

def generar_texto(palabras_usuario):
    prompt_entrada = f"<start_of_turn>user\nCrea una oración con: {palabras_usuario}.<end_of_turn>\n<start_of_turn>model\nOración:"
    entradas = tokenizador([prompt_entrada], return_tensors="pt").to("cuda")

    salidas = modelo.generate(
        **entradas,
        max_new_tokens=25,
        max_length=None,
        do_sample=False,
        repetition_penalty=1.0,
        eos_token_id=tokenizador.eos_token_id
    )

    texto_decodificado = tokenizador.decode(salidas[0][entradas.input_ids.shape[1]:], skip_special_tokens=True)
    oracion_limpia = texto_decodificado.strip().split('.')[0]
    return oracion_limpia + "." if oracion_limpia else "Error al generar."

print("\n" + "="*50)
print("💬 CHAT INTERACTIVO Y AUTOGUARDADO")
print("1. Escribe tus palabras clave.")
print("2. Presiona ENTER para aceptar la oración generada.")
print("3. Escribe tu propia oración si deseas corregirla.")
print("4. Escribe 'salir' para finalizar y cerrar.")
print("="*50 + "\n")

while True:
    try:
        entrada_palabras = input("🔹 Palabras: ").strip()
    except (KeyboardInterrupt, EOFError):
        break

    if entrada_palabras.lower() == "salir":
        print(f"\n👋 ¡Adiós! Progreso guardado exitosamente en '{ruta_local}'.")
        break
    if not entrada_palabras:
        continue

    respuesta_modelo = generar_texto(entrada_palabras)
    print(f"   🤖 Respuesta: {respuesta_modelo}")

    try:
        correccion_usuario = input("   ✍️ Corrección (Enter si es correcta): ").strip()
    except (KeyboardInterrupt, EOFError):
        break

    oracion_definitiva = correccion_usuario if correccion_usuario else respuesta_modelo
    
    lista_datos.append({"palabras": entrada_palabras, "oracion": oracion_definitiva})

    with open(ruta_local, "w", encoding="utf-8") as f:
        json.dump({"root": lista_datos}, f, ensure_ascii=False, indent=4)

    if correccion_usuario:
        print(f"   [✔️ Corregido y almacenado en {ruta_local}]\n")
    else:
        print(f"   [✅ Aprobado y almacenado en {ruta_local}]\n")

📂 Iniciando carga de datos...
¡Cargados 63 ejemplos desde el dataset original de Kaggle!
Total de ejemplos listos para el entrenamiento: 63

🤖 Cargando modelo base Gemma 3 1B...
==((====))==  Unsloth 2026.9.2: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
total weights      : 1.862 GiB
no_split classes   : ['Gemma3DecoderLayer', 'SiglipEncoderLayer', 'SiglipMultiheadAttentionPoolingHead', 'SiglipVisionEmbeddings']
output head        : lm_head -> cuda:1
head headroom      : 0.438 GiB
activation reserve : 12.211 GiB requested
tied to head       : ['model

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

⚙️ Configurando el adaptador LoRA...
Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


Map:   0%|          | 0/63 [00:00<?, ? examples/s]

Unsloth: Switching to float32 training since model cannot work with float16


...


Enlaces alternativos de descarga (FileLink):


/kaggle/working/modelo_gemma_gguf.zip

/kaggle/working/modelo_gemma_litert.zip


🔄 Activando modo de inferencia rápida...

💬 CHAT INTERACTIVO Y AUTOGUARDADO
1. Escribe tus palabras clave.
2. Presiona ENTER para aceptar la oración generada.
3. Escribe tu propia oración si deseas corregirla.
4. Escribe 'salir' para finalizar y cerrar.



🔹 Palabras:  hola
